# FIT3D Test Set Inference & Submission

Runs the MediaPipe + TCN pipeline on the FIT3D **test** split (subjects s02, s12, s13) and
generates `fit3d_submission.json` ready for competition upload.

Normalization and architecture are **identical** to `medipipe-3d-with-tcn.ipynb`.

## 1. Imports

In [1]:
import os
import json
import numpy as np
import cv2
import mediapipe as mp
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from pathlib import Path
from tqdm import tqdm

## 2. Paths Configuration

In [2]:
dataset_Folder   = "G:/My Drive/GP Datasets/fit3d_data/"
template_path    = dataset_Folder + "fit3d_template.json"
test_subj_names  = ["s02", "s12", "s13"]

# Model path – tries Kaggle working dir first, then local
_kaggle_model = "/kaggle/working/best_tcn_model.pt"
_local_model  = "../../Models/best_tcn_model.pt"
model_path = (
    _kaggle_model if Path(_kaggle_model).exists()
    else _local_model  if Path(_local_model).exists()
    else "best_tcn_model.pt"
)
print(f"Using model: {model_path}")

output_dir = "D:/GP/Pose/Outputs"
SEQ_LEN    = 81   # must match training

Using model: ../../Models/best_tcn_model.pt


## 3. TCN Architecture  *(identical to training notebook)*

In [3]:
class TemporalConv(nn.Module):
    def __init__(self, in_channels, out_channels, kernel_size, dilation, dropout=0.25):
        super().__init__()
        padding = (kernel_size - 1) * dilation // 2
        self.conv    = nn.Conv1d(in_channels, out_channels, kernel_size, dilation=dilation, padding=padding)
        self.bn      = nn.BatchNorm1d(out_channels)
        self.relu    = nn.ReLU(inplace=True)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        return self.dropout(self.relu(self.bn(self.conv(x))))


class ResidualBlock1D(nn.Module):
    def __init__(self, channels, kernel_size, dilation, dropout=0.25):
        super().__init__()
        self.conv1   = TemporalConv(channels, channels, kernel_size, dilation, dropout)
        self.conv2   = nn.Conv1d(channels, channels, kernel_size, dilation=dilation,
                                 padding=(kernel_size - 1) * dilation // 2)
        self.bn      = nn.BatchNorm1d(channels)
        self.relu    = nn.ReLU(inplace=True)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        res = self.conv1(x)
        res = self.dropout(self.bn(self.conv2(res)))
        return self.relu(x + res)


class Temporal3DRefinementNet(nn.Module):
    """
    Maps (B, T, 33, 3) MediaPipe windows → (B, T, 25, 3) Fit3D predictions.
    Architecture identical to the training notebook.
    """
    def __init__(self, num_joints_in=33, in_features=3,
                 num_joints_out=25, out_features=3, channels=256, dropout=0.25):
        super().__init__()
        self.in_dim  = num_joints_in  * in_features
        self.out_dim = num_joints_out * out_features

        self.expand = TemporalConv(self.in_dim, channels, kernel_size=1, dilation=1, dropout=0.0)

        dilations = [1, 3, 9, 27]
        self.res_blocks = nn.ModuleList([
            ResidualBlock1D(channels, kernel_size=3, dilation=d, dropout=dropout)
            for d in dilations
        ])

        self.shrink = nn.Conv1d(channels, self.out_dim, kernel_size=1)

    def forward(self, x):
        B, T, V_in, C_in = x.shape
        x = x.view(B, T, -1).permute(0, 2, 1)   # (B, V_in*C_in, T)
        x = self.expand(x)
        for block in self.res_blocks:
            x = block(x)
        x = self.shrink(x)                        # (B, V_out*C_out, T)
        x = x.view(B, self.out_dim // 3, 3, T).permute(0, 3, 1, 2)  # (B, T, 25, 3)
        return x

## 4. Per-Video Sliding-Window Dataset  *(identical to training notebook)*

In [4]:
class SingleVideoDataset(Dataset):
    """
    Builds sliding windows for one video.  Accepts a dummy gt_s (zeros)
    so the DataLoader API matches the training notebook; only pred_t is
    used during inference.
    """
    def __init__(self, pred_s, gt_s, seq_len=81):
        self.pred_s   = pred_s
        self.gt_s     = gt_s
        self.seq_len  = seq_len
        self.half_seq = seq_len // 2
        self.min_len  = min(len(pred_s), len(gt_s))

    def __len__(self):
        return self.min_len

    def __getitem__(self, idx):
        half      = self.half_seq
        raw_start = idx - half
        raw_end   = idx + half + 1
        h5_start  = max(0, raw_start)
        h5_end    = min(self.min_len, raw_end)

        pred_sub = self.pred_s[h5_start:h5_end]
        gt_sub   = self.gt_s  [h5_start:h5_end]

        pad_left  = h5_start - raw_start
        pad_right = raw_end  - h5_end

        pred_t = torch.from_numpy(pred_sub.astype(np.float32))
        gt_t   = torch.from_numpy(gt_sub  .astype(np.float32))

        if pad_left > 0:
            pred_t = torch.cat([pred_t[0:1].expand(pad_left, -1, -1), pred_t], dim=0)
            gt_t   = torch.cat([gt_t  [0:1].expand(pad_left, -1, -1), gt_t  ], dim=0)
        if pad_right > 0:
            pred_t = torch.cat([pred_t, pred_t[-1:].expand(pad_right, -1, -1)], dim=0)
            gt_t   = torch.cat([gt_t,   gt_t  [-1:].expand(pad_right, -1, -1)], dim=0)

        return pred_t, gt_t

## 5. MediaPipe + TCN Inference Function

Normalization is **identical** to the evaluation cell of `medipipe-3d-with-tcn.ipynb`:
- Root-relative: subtract midpoint of joints **23** (L_Hip) and **24** (R_Hip)
- Scale: divide by median norm of shoulders midpoint (joints **11** & **12**)

In [5]:
def predict_mediapipe_tcn(video_path, model, device, seq_len=81):
    """
    Run MediaPipe → TCN on a single video.

    Returns
    -------
    pred_3d_25 : np.ndarray  shape (N, 25, 3)  in metres (normalised-space × spine-scale)
                 or None if MediaPipe extracts no frames.
    """
    # ---- 1. MediaPipe extraction ------------------------------------------------
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        return None

    pred_3d_list = []
    last_pose    = np.zeros((33, 3), dtype=np.float32)

    mp_pose = mp.solutions.pose
    with mp_pose.Pose(min_detection_confidence=0.5,
                      min_tracking_confidence=0.5,
                      model_complexity=1,
                      static_image_mode=False) as pose:
        while cap.isOpened():
            ret, frame = cap.read()
            if not ret:
                break
            image = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            image.flags.writeable = False
            results = pose.process(image)

            if results and getattr(results, 'pose_world_landmarks', None):
                current_pose = np.array(
                    [[lm.x, lm.y, lm.z] for lm in results.pose_world_landmarks.landmark],
                    dtype=np.float32
                )
            else:
                current_pose = last_pose

            pred_3d_list.append(current_pose)
            last_pose = current_pose

    cap.release()

    if len(pred_3d_list) == 0:
        return None

    pred_3d = np.array(pred_3d_list, dtype=np.float32)   # (N, 33, 3)
    N       = len(pred_3d)

    # ---- 2. Sliding-window TCN inference ----------------------------------------
    # Dummy GT (zeros) so DataLoader API matches training notebook
    dummy_gt = np.zeros((N, 25, 3), dtype=np.float32)

    half_seq = seq_len // 2
    video_dataset = SingleVideoDataset(pred_3d, dummy_gt, seq_len=seq_len)
    video_loader  = DataLoader(video_dataset, batch_size=2048, pin_memory=True, shuffle=False, num_workers=0)

    all_preds       = []   # (N, 25, 3) – one entry per centre frame

    with torch.no_grad():
        for window, _ in video_loader:
            window = window.to(device, non_blocking=True)   # (B, T, 33, 3)

            # ---- Normalization (IDENTICAL to training/evaluation cell) ------------
            # Root-relative: joints 23 (L_Hip) and 24 (R_Hip)
            mp_root = ((window[:, :, 23, :] + window[:, :, 24, :]) / 2.0).unsqueeze(2)
            window  = window - mp_root

            # Sequence-level scale: midpoint of joints 11 (L_Shoulder) & 12 (R_Shoulder)
            mp_shoulders        = (window[:, :, 11, :] + window[:, :, 12, :]) / 2.0
            mp_spine_len_all    = torch.norm(mp_shoulders, dim=-1, keepdim=True).unsqueeze(-1)
            mp_spine_len_median = mp_spine_len_all.median(dim=1, keepdim=True)[0]   # (B,1,1,1)
            window              = window / mp_spine_len_median.clamp(min=1e-5)
            # -----------------------------------------------------------------------

            pred = model(window)              # (B, T, 25, 3)  – normalised Fit3D space
            pred_mid = pred[:, half_seq]      # (B, 25, 3)  centre frame

            # De-normalise back to metres using the MediaPipe spine scale
            scale = mp_spine_len_median[:, 0, 0, 0]  # (B,)
            pred_mid_denorm = pred_mid * scale.view(-1, 1, 1)  # (B, 25, 3)

            all_preds.append(pred_mid_denorm.cpu().numpy())

    pred_3d_25 = np.concatenate(all_preds, axis=0)  # (N, 25, 3) metres
    return pred_3d_25

## 6. Fit3D 25-Joint → H36M 17-Joint Mapping

**Fit3D 25-joint skeleton** (from FIT3D_BONES in training notebook):
```
0:Pelvis  1:L_Hip  2:L_Knee  3:L_Ankle
4:R_Hip   5:R_Knee 6:R_Ankle
7:Spine_lower  9:Spine_upper/Thorax  10:Neck
11:L_Shoulder 12:L_Elbow 13:L_Wrist
14:R_Shoulder 15:R_Elbow 16:R_Wrist
17,19,21:L_Hand  18,20,22:R_Hand
```

**H36M 17-joint skeleton**:
```
0:Pelvis  1:R_Hip  2:R_Knee  3:R_Ankle
4:L_Hip   5:L_Knee 6:L_Ankle
7:Spine/Thorax  8:Neck  9:Head  10:HeadTop
11:L_Shoulder 12:L_Elbow 13:L_Wrist
14:R_Shoulder 15:R_Elbow 16:R_Wrist
```

In [6]:
def map_fit3d25_to_h36m17(pred_3d_25):
    """
    Maps (N, 25, 3) Fit3D predictions → (N, 17, 3) H36M predictions.

    Fit3D joint indices match the FIT3D_BONES definition in the TCN training notebook.
    H36M indices follow the standard 17-joint layout used by the FIT3D competition.
    """
    if pred_3d_25 is None:
        return None

    N = pred_3d_25.shape[0]
    mapped = np.zeros((N, 17, 3), dtype=np.float32)

    # Fit3D indices (from FIT3D_BONES in the TCN notebook)
    Pelvis      = 0
    L_Hip       = 1;  L_Knee    = 2;  L_Ankle   = 3
    R_Hip       = 4;  R_Knee    = 5;  R_Ankle   = 6
    Spine_upper = 9   # (7→9→10 in FIT3D_BONES = Thorax)
    Neck        = 10
    L_Shoulder  = 11; L_Elbow   = 12; L_Wrist   = 13
    R_Shoulder  = 14; R_Elbow   = 15; R_Wrist   = 16

    # H36M 0: Pelvis
    mapped[:, 0]  = pred_3d_25[:, Pelvis]
    # H36M 1-3: Right leg
    mapped[:, 1]  = pred_3d_25[:, R_Hip]
    mapped[:, 2]  = pred_3d_25[:, R_Knee]
    mapped[:, 3]  = pred_3d_25[:, R_Ankle]
    # H36M 4-6: Left leg
    mapped[:, 4]  = pred_3d_25[:, L_Hip]
    mapped[:, 5]  = pred_3d_25[:, L_Knee]
    mapped[:, 6]  = pred_3d_25[:, L_Ankle]
    # H36M 7: Spine/Thorax  → Fit3D Spine_upper (index 9)
    mapped[:, 7]  = pred_3d_25[:, Spine_upper]
    # H36M 8: Neck  → Fit3D Neck (index 10)
    mapped[:, 8]  = pred_3d_25[:, Neck]
    # H36M 9: Head (approx.: half-step above Neck along spine direction)
    spine_dir     = pred_3d_25[:, Neck] - pred_3d_25[:, Spine_upper]
    mapped[:, 9]  = pred_3d_25[:, Neck] + spine_dir * 0.5
    # H36M 10: Head Top (another half-step beyond Head)
    mapped[:, 10] = mapped[:, 9] + spine_dir * 0.5
    # H36M 11-13: Left arm
    mapped[:, 11] = pred_3d_25[:, L_Shoulder]
    mapped[:, 12] = pred_3d_25[:, L_Elbow]
    mapped[:, 13] = pred_3d_25[:, L_Wrist]
    # H36M 14-16: Right arm
    mapped[:, 14] = pred_3d_25[:, R_Shoulder]
    mapped[:, 15] = pred_3d_25[:, R_Elbow]
    mapped[:, 16] = pred_3d_25[:, R_Wrist]

    return mapped

## 7. Test Task Discovery

In [7]:
def get_test_tasks(dataset_base, subj_names, template_data):
    """
    Build a list of test tasks from the template.
    Each task: {subj, action, video_path, req_frames}
    Video paths: <dataset_base>/test/<subj>/videos/<action>.mp4
    """
    tasks = []
    if not template_data:
        return tasks
    for subj, actions in template_data.items():
        if subj not in subj_names:
            continue
        for action, action_data in actions.items():
            video_path = os.path.join(dataset_base, "test", subj, "videos", f"{action}.mp4")
            if os.path.exists(video_path):
                req_frames = action_data.get("other", {}).get("video_fr_ids", [])
                tasks.append({
                    'subj'      : subj,
                    'action'    : action,
                    'video_path': video_path,
                    'req_frames': req_frames,
                })
    return tasks

## 8. Test Evaluation Pipeline

In [8]:
def evaluate_test_set(tasks, model, device, template_data, out_dir, seq_len=81, save_interval=10, max_workers=4):
    """
    For each test video:
      1. Run MediaPipe + TCN → (N, 25, 3) metres
      2. Convert to mm, map to H36M 17-joint format
      3. Extract the required frames specified by the template
      4. Write results into template_data and save JSON
    """
    from concurrent.futures import ThreadPoolExecutor, as_completed

    test_results = {}   # {subj: {action: frames_data}}
    completed_count = 0

    def process_task(i, t):
        pred_3d_25 = predict_mediapipe_tcn(t['video_path'], model, device, seq_len=seq_len)
        return i, t, pred_3d_25

    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        futures = {executor.submit(process_task, i, t): (i, t) for i, t in enumerate(tasks)}

        for future in tqdm(as_completed(futures), total=len(tasks), desc="Test inference (Parallel)"):
            i, t = futures[future]
            try:
                _, _, pred_3d_25 = future.result()
            except Exception as e:
                print(f"  WARN: Error processing {t['subj']}/{t['action']}: {e}", flush=True)
                continue

            subj   = t['subj']
            action = t['action']

            if pred_3d_25 is None:
                print(f"  WARN: No prediction for {subj}/{action}", flush=True)
                continue

            # Convert metres → millimetres
            pred_3d_25_mm = pred_3d_25 * 1000.0

            # Map Fit3D 25 joints → H36M 17 joints
            pred_3d_17 = map_fit3d25_to_h36m17(pred_3d_25_mm)   # (N, 17, 3)

            # Extract the frames required by the template
            max_f = pred_3d_17.shape[0] - 1
            frames_data = [
                pred_3d_17[min(f, max_f)].tolist()
                for f in t['req_frames']
            ]

            if subj not in test_results:
                test_results[subj] = {}
            test_results[subj][action] = frames_data

            completed_count += 1
            # ---- Checkpoint save -----------------------------------------------------
            if completed_count % save_interval == 0:
                for s_tmp, acts_tmp in test_results.items():
                    if s_tmp not in template_data: continue
                    for act_tmp, f_data_tmp in acts_tmp.items():
                        if act_tmp not in template_data[s_tmp]: continue
                        persons = template_data[s_tmp][act_tmp].get("persons", [])
                        if len(persons) > 0:
                            if "joints3d" not in persons[0]:
                                persons[0]["joints3d"] = {}
                            persons[0]["joints3d"]["joints3d"] = f_data_tmp
                os.makedirs(out_dir, exist_ok=True)
                chk_out = os.path.join(out_dir, "fit3d_submission_checkpoint.json")
                with open(chk_out, 'w') as f_chk:
                    json.dump(template_data, f_chk)

    # ---- Populate template -------------------------------------------------------
    for subj, actions in test_results.items():
        if subj not in template_data:
            continue
        for action, frames_data in actions.items():
            if action not in template_data[subj]:
                continue
            persons = template_data[subj][action].get("persons", [])
            if len(persons) > 0:
                if "joints3d" not in persons[0]:
                    persons[0]["joints3d"] = {}
                persons[0]["joints3d"]["joints3d"] = frames_data

    os.makedirs(out_dir, exist_ok=True)
    final_out = os.path.join(out_dir, "fit3d_submission.json")
    with open(final_out, 'w') as f:
        json.dump(template_data, f)
    print(f"Submission saved to: {final_out}")


## 9. Run Everything

In [9]:
# ---- Load template (or checkpoint if exists) ------------------------------------
assert os.path.exists(template_path), f"Template not found: {template_path}"
with open(template_path, 'r') as f:
    template_data = json.load(f)

chk_path = os.path.join(output_dir, "fit3d_submission_checkpoint.json")
done_keys = set()
if os.path.exists(chk_path):
    print(f"Checkpoint found: {chk_path}  – resuming from checkpoint.", flush=True)
    with open(chk_path, 'r') as f:
        chk_data = json.load(f)
    # Merge checkpoint predictions into template_data
    for subj, actions in chk_data.items():
        if subj not in template_data:
            continue
        for action, action_data in actions.items():
            if action not in template_data[subj]:
                continue
            persons_chk = action_data.get("persons", [])
            persons_tpl = template_data[subj][action].get("persons", [])
            # Check if this action was already predicted in the checkpoint
            if (len(persons_chk) > 0
                    and "joints3d" in persons_chk[0]
                    and persons_chk[0]["joints3d"].get("joints3d")):
                if len(persons_tpl) > 0:
                    persons_tpl[0]["joints3d"] = persons_chk[0]["joints3d"]
                done_keys.add((subj, action))
    print(f"Skipping {len(done_keys)} already-processed videos.", flush=True)
else:
    print("No checkpoint found – starting fresh.", flush=True)

print(f"Template loaded. Subjects in template: {list(template_data.keys())}")

# ---- Load TCN model -------------------------------------------------------------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

model = Temporal3DRefinementNet(num_joints_in=33, num_joints_out=25).to(device)
assert os.path.exists(model_path), f"Model checkpoint not found: {model_path}"
state_dict = torch.load(model_path, map_location=device, weights_only=True)
# Strip torch.compile prefix if present
clean_sd = {k.replace('_orig_mod.', ''): v for k, v in state_dict.items()}
model.load_state_dict(clean_sd)
model.eval()
print(f"TCN model loaded from: {model_path}")

# ---- Build test tasks (skip already-done) ----------------------------------------
all_test_tasks = get_test_tasks(dataset_Folder, test_subj_names, template_data)
test_tasks = [t for t in all_test_tasks
              if (t['subj'], t['action']) not in done_keys]
print(f"Found {len(all_test_tasks)} total test videos; "
      f"{len(done_keys)} already done, {len(test_tasks)} remaining.")

# ---- Inference & submission -----------------------------------------------------
if test_tasks:
    evaluate_test_set(test_tasks, model, device, template_data, output_dir, seq_len=SEQ_LEN)
else:
    # All done – just write out the final file from the merged template
    os.makedirs(output_dir, exist_ok=True)
    final_out = os.path.join(output_dir, "fit3d_submission.json")
    with open(final_out, 'w') as f:
        json.dump(template_data, f)
    print(f"All videos already processed. Final submission written to: {final_out}")


Checkpoint found: D:/GP/Pose/Outputs\fit3d_submission_checkpoint.json  – resuming from checkpoint.
Skipping 141 already-processed videos.
Template loaded. Subjects in template: ['s02', 's12', 's13']
Device: cuda
TCN model loaded from: ../../Models/best_tcn_model.pt
Found 141 total test videos; 141 already done, 0 remaining.
All videos already processed. Final submission written to: D:/GP/Pose/Outputs\fit3d_submission.json
